# Copilot M365 Activity — Usage Report Ingester (Fabric)

Loads **daily Microsoft 365 activity per person** from the Microsoft Graph usage reports, so ValueLens can show how people work across Microsoft 365 and where Copilot fits in.

| Report | Graph function | What it adds |
|---|---|---|
| Teams | `getTeamsUserActivityUserDetail` | Chats, calls, meetings, channel posts, call minutes |
| Outlook | `getEmailActivityUserDetail` | Emails sent, received and read, meetings created |
| SharePoint | `getSharePointActivityUserDetail` | Files viewed, synced and shared, pages visited |
| OneDrive | `getOneDriveActivityUserDetail` | Files viewed, synced and shared |
| Viva Engage | `getYammerActivityUserDetail` | Posts, reads and likes |
| Microsoft 365 apps | `getM365AppUserDetail` | Word, Excel, PowerPoint, OneNote, Outlook and Teams use, by platform |

**How it loads**: Microsoft keeps the last 28 days of daily reports. The first run loads every day still available. Later runs load the days not yet in the table and re-read the most recent days (`REREAD_DAYS`), which Microsoft may still be filling in. Each day is replaced as a whole, so re-running never duplicates rows. History builds up from there and is kept unless `RETAIN_DAYS` is set.

**Output**: Lakehouse Delta table `dbo.m365_activity_daily`, one row per person per day.

**Permissions**: app registration with `Reports.Read.All` (Application permission, admin-consented) — the same permission the audit log ingester already uses.

**Names in reports**: if the Microsoft 365 admin center conceals user names in reports, people come back as random IDs and cannot be matched to Copilot use. Turn it off in **Microsoft 365 admin center → Settings → Org settings → Reports**. The notebook warns when it sees concealed names, and reloads those days with real names once the setting is off.

## 1. Configuration

In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'

OUTPUT_TABLE  = 'dbo.m365_activity_daily'
REREAD_DAYS   = 4    # re-read the most recent days, which Microsoft may still be filling in
RETAIN_DAYS   = 0    # 0 keeps every day; e.g. 400 deletes days older than that

## 2. Authenticate to Microsoft Graph

In [ ]:
import time
import requests

GRAPH = 'https://graph.microsoft.com/v1.0'


def get_graph_token():
    r = requests.post(
        f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
        data={
            'client_id':     CLIENT_ID,
            'scope':         'https://graph.microsoft.com/.default',
            'client_secret': CLIENT_SECRET,
            'grant_type':    'client_credentials',
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json()['access_token']


_token = {'value': get_graph_token()}
print('✓ Graph token acquired.')

## 3. Read one usage report

Each report is a CSV for a single day. Graph answers with a redirect to a short-lived download link, which `requests` follows without sending the token on.

- **400 / 404**: Microsoft has no report for that day (outside the 28-day window, or not ready yet).
- **401**: the token expired — get a new one and retry once.
- **403**: the app registration is missing `Reports.Read.All` — stop with a clear message.
- **429 / 5xx**: wait (honouring `Retry-After`) and retry.

In [ ]:
import csv
import io

REPORTS = {
    'teams':      'getTeamsUserActivityUserDetail',
    'email':      'getEmailActivityUserDetail',
    'sharepoint': 'getSharePointActivityUserDetail',
    'onedrive':   'getOneDriveActivityUserDetail',
    'vivaengage': 'getYammerActivityUserDetail',
    'apps':       'getM365AppUserDetail',
}


class ReportUnavailable(Exception):
    """Microsoft has no report for this day."""


def _retry_after(response, attempt):
    try:
        return max(1, min(120, int(float(response.headers.get('Retry-After', '')))))
    except ValueError:
        return min(60, 2 ** attempt)


def fetch_report(function, day, attempts=6):
    url = f'{GRAPH}/reports/{function}(date={day.isoformat()})'
    refreshed = False
    for attempt in range(1, attempts + 1):
        try:
            r = requests.get(url, headers={'Authorization': f"Bearer {_token['value']}"}, timeout=180)
        except requests.RequestException:
            if attempt == attempts:
                raise
            time.sleep(min(60, 2 ** attempt))
            continue
        if r.status_code == 200:
            return list(csv.DictReader(io.StringIO(r.content.decode('utf-8-sig'))))
        if r.status_code == 401 and not refreshed:
            _token['value'] = get_graph_token()
            refreshed = True
            continue
        if r.status_code == 403:
            raise PermissionError(
                f'Graph returned 403 for {function}. Give the app registration the Reports.Read.All '
                'application permission with admin consent, then run again.'
            )
        if r.status_code in (400, 404):
            raise ReportUnavailable(f'{function} {day}: HTTP {r.status_code}')
        if (r.status_code == 429 or r.status_code >= 500) and attempt < attempts:
            time.sleep(_retry_after(r, attempt))
            continue
        r.raise_for_status()
    raise RuntimeError(f'{function} {day}: no answer after {attempts} attempts.')

## 4. Shape the reports into one row per person per day

Columns are read by name and default to 0, so a column Microsoft adds, renames or drops never breaks the load. The `…Active` flags say whether the person did something in that workload that day; receiving email on its own does not count. `AppsActive` means Word, Excel, PowerPoint or OneNote was used. `WorkloadsActive` counts the active flags (0–6).

In [ ]:
import re
from datetime import datetime, timedelta, timezone

# (output column, report, CSV columns to try in order)
COUNTS = [
    ('TeamsChatMessages',               'teams',      ['Team Chat Message Count']),
    ('TeamsPrivateChatMessages',        'teams',      ['Private Chat Message Count']),
    ('TeamsCalls',                      'teams',      ['Call Count']),
    ('TeamsMeetings',                   'teams',      ['Meeting Count']),
    ('TeamsMeetingsOrganized',          'teams',      ['Meetings Organized Count']),
    ('TeamsMeetingsAttended',           'teams',      ['Meetings Attended Count']),
    ('TeamsPostMessages',               'teams',      ['Post Messages']),
    ('TeamsReplyMessages',              'teams',      ['Reply Messages']),
    ('TeamsAudioSeconds',               'teams',      ['Audio Duration In Seconds', 'Audio Duration']),
    ('TeamsVideoSeconds',               'teams',      ['Video Duration In Seconds', 'Video Duration']),
    ('TeamsScreenShareSeconds',         'teams',      ['Screen Share Duration In Seconds', 'Screen Share Duration']),
    ('EmailSent',                       'email',      ['Send Count']),
    ('EmailReceived',                   'email',      ['Receive Count']),
    ('EmailRead',                       'email',      ['Read Count']),
    ('EmailMeetingsCreated',            'email',      ['Meeting Created Count']),
    ('EmailMeetingsInteracted',         'email',      ['Meeting Interacted Count']),
    ('SharePointFilesViewedOrEdited',   'sharepoint', ['Viewed Or Edited File Count']),
    ('SharePointFilesSynced',           'sharepoint', ['Synced File Count']),
    ('SharePointFilesSharedInternally', 'sharepoint', ['Shared Internally File Count']),
    ('SharePointFilesSharedExternally', 'sharepoint', ['Shared Externally File Count']),
    ('SharePointPagesVisited',          'sharepoint', ['Visited Page Count']),
    ('OneDriveFilesViewedOrEdited',     'onedrive',   ['Viewed Or Edited File Count']),
    ('OneDriveFilesSynced',             'onedrive',   ['Synced File Count']),
    ('OneDriveFilesSharedInternally',   'onedrive',   ['Shared Internally File Count']),
    ('OneDriveFilesSharedExternally',   'onedrive',   ['Shared Externally File Count']),
    ('VivaEngagePosts',                 'vivaengage', ['Posted Count']),
    ('VivaEngageReads',                 'vivaengage', ['Read Count']),
    ('VivaEngageLikes',                 'vivaengage', ['Liked Count']),
]
FLAGS = [
    ('AppOutlook',      'apps', ['Outlook']),
    ('AppWord',         'apps', ['Word']),
    ('AppExcel',        'apps', ['Excel']),
    ('AppPowerPoint',   'apps', ['PowerPoint']),
    ('AppOneNote',      'apps', ['OneNote']),
    ('AppTeams',        'apps', ['Teams']),
    ('PlatformWindows', 'apps', ['Windows']),
    ('PlatformMac',     'apps', ['Mac']),
    ('PlatformMobile',  'apps', ['Mobile']),
    ('PlatformWeb',     'apps', ['Web']),
]
ACTIVE = {
    'TeamsActive':      ['TeamsChatMessages', 'TeamsPrivateChatMessages', 'TeamsCalls', 'TeamsMeetings',
                         'TeamsMeetingsOrganized', 'TeamsMeetingsAttended', 'TeamsPostMessages', 'TeamsReplyMessages'],
    'EmailActive':      ['EmailSent', 'EmailRead', 'EmailMeetingsCreated', 'EmailMeetingsInteracted'],
    'SharePointActive': [c for c, r, _ in COUNTS if r == 'sharepoint'],
    'OneDriveActive':   [c for c, r, _ in COUNTS if r == 'onedrive'],
    'VivaEngageActive': [c for c, r, _ in COUNTS if r == 'vivaengage'],
    'AppsActive':       ['AppWord', 'AppExcel', 'AppPowerPoint', 'AppOneNote'],
}
VALUE_COLUMNS = [c for c, _, _ in COUNTS] + [c for c, _, _ in FLAGS]

_ISO_DURATION = re.compile(r'^P(?:(\d+(?:\.\d+)?)D)?(?:T(?:(\d+(?:\.\d+)?)H)?(?:(\d+(?:\.\d+)?)M)?(?:(\d+(?:\.\d+)?)S)?)?$')


def _number(value):
    text = (value or '').strip()
    if not text:
        return 0
    try:
        return int(float(text))
    except ValueError:
        m = _ISO_DURATION.match(text.upper())
        if not m:
            return 0
        d, h, mi, s = (float(g or 0) for g in m.groups())
        return int(d * 86400 + h * 3600 + mi * 60 + s)


def _flag(value):
    return 1 if (value or '').strip().lower() in ('yes', 'true', '1') else 0


def _date(value):
    try:
        return datetime.strptime((value or '').strip()[:10], '%Y-%m-%d').date()
    except ValueError:
        return None


def _pick(record, names):
    for name in names:
        if name.lower() in record:
            return record[name.lower()]
    return ''


BY_REPORT = {
    key: ([(c, n) for c, r, n in COUNTS if r == key], [(c, n) for c, r, n in FLAGS if r == key])
    for key in REPORTS
}


def add_report(people, key, records):
    counts, flags = BY_REPORT[key]
    for raw in records:
        record = {(k or '').strip().lower(): v for k, v in raw.items()}
        upn = (record.get('user principal name') or '').strip()
        if not upn:
            continue
        person = people.setdefault(upn.lower(), {'UPN': upn, 'ReportRefreshDate': None})
        refreshed = _date(record.get('report refresh date'))
        if refreshed and (person['ReportRefreshDate'] is None or refreshed > person['ReportRefreshDate']):
            person['ReportRefreshDate'] = refreshed
        for column, names in counts:
            person[column] = person.get(column, 0) + _number(_pick(record, names))
        for column, names in flags:
            person[column] = max(person.get(column, 0), _flag(_pick(record, names)))

## 5. Decide which days to load

Days still in Microsoft's window that are not in the table yet, the last `REREAD_DAYS` days, and any day loaded while names were concealed.

In [ ]:
from pyspark.sql import functions as F


def days_to_load(today, loaded, concealed_days, reread_days, window_days=28):
    recent = today - timedelta(days=reread_days)
    window = [today - timedelta(days=n) for n in range(window_days, 0, -1)]
    return [day for day in window if day not in loaded or day >= recent or day in concealed_days]


today = datetime.now(timezone.utc).date()
loaded, concealed_days = set(), set()
if spark.catalog.tableExists(OUTPUT_TABLE):
    by_day = (spark.table(OUTPUT_TABLE)
              .groupBy('ActivityDate')
              .agg(F.avg(F.when(F.col('UPN').contains('@'), 0).otherwise(1)).alias('concealed'))
              .collect())
    loaded = {r['ActivityDate'] for r in by_day}
    concealed_days = {r['ActivityDate'] for r in by_day if r['concealed'] > 0.5}
todo = days_to_load(today, loaded, concealed_days, REREAD_DAYS)
print(f'{len(loaded):,} days already loaded. Loading {len(todo)} day(s): '
      f'{todo[0] if todo else "-"} to {todo[-1] if todo else "-"}.')

## 6. Load each day

The six reports for a day are read together. Each day is written on its own and replaces whatever was there for that day, so the load is safe to re-run.

- A report Microsoft has no data for adds nothing; the rest of the day still loads.
- When re-reading a day that is already loaded, every report must answer, so a gap never overwrites good data.
- Any other error skips the day; the next run tries it again.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql.types import (DateType, IntegerType, LongType, StringType, StructField, StructType,
                               TimestampType)

SCHEMA = StructType(
    [
        StructField('ActivityDate', DateType(), False),
        StructField('WeekStart', DateType(), False),
        StructField('UPN', StringType(), False),
        StructField('UPN_Normalized', StringType(), False),
        StructField('ReportRefreshDate', DateType(), True),
    ]
    + [StructField(c, LongType(), False) for c, _, _ in COUNTS]
    + [StructField(c, IntegerType(), False) for c, _, _ in FLAGS]
    + [StructField(c, IntegerType(), False) for c in ACTIVE]
    + [
        StructField('WorkloadsActive', IntegerType(), False),
        StructField('LoadedAtUtc', TimestampType(), False),
    ]
)


def build_rows(day, people, loaded_at):
    week_start = day - timedelta(days=day.weekday())
    rows = []
    for key, person in people.items():
        values = [person.get(c, 0) for c in VALUE_COLUMNS]
        if not any(values):
            continue
        active = [1 if any(person.get(c, 0) for c in cols) else 0 for cols in ACTIVE.values()]
        rows.append(tuple([day, week_start, person['UPN'], key, person['ReportRefreshDate']]
                          + values + active + [sum(active), loaded_at]))
    return rows


def write_day(day, rows):
    df = spark.createDataFrame(rows, schema=SCHEMA)
    writer = df.write.format('delta').mode('overwrite').option('mergeSchema', 'true')
    if spark.catalog.tableExists(OUTPUT_TABLE):
        writer = writer.option('replaceWhere', f"ActivityDate = DATE '{day.isoformat()}'")
    writer.saveAsTable(OUTPUT_TABLE)


def read_day(day):
    def one(key):
        try:
            return key, fetch_report(REPORTS[key], day)
        except ReportUnavailable:
            return key, None

    with ThreadPoolExecutor(max_workers=3) as pool:
        return dict(pool.map(one, REPORTS))


written, skipped, people_seen, concealed_seen = [], [], 0, 0
for day in todo:
    rereading = day in loaded
    try:
        reports = read_day(day)
    except PermissionError:
        raise
    except Exception as error:
        skipped.append((day, f'{type(error).__name__}: {error}'))
        print(f'  ! {day}: skipped ({type(error).__name__}: {error})')
        continue
    missing = sorted(k for k, records in reports.items() if records is None)
    if rereading and missing:
        print(f'  - {day}: kept the loaded day ({", ".join(missing)} not available on re-read)')
        continue
    people = {}
    for key, records in reports.items():
        if records:
            add_report(people, key, records)
    rows = build_rows(day, people, datetime.now(timezone.utc).replace(tzinfo=None))
    note = f' ({", ".join(missing)} not available)' if missing else ''
    if not rows:
        print(f'  - {day}: no activity reported yet{note}')
        continue
    write_day(day, rows)
    written.append(day)
    if len(rows) >= people_seen:
        people_seen = len(rows)
        concealed_seen = sum(1 for r in rows if '@' not in r[2])
    print(f'  ✓ {day}: {len(rows):,} people{note}')

if todo and not written and len(skipped) == len(todo):
    raise RuntimeError(f'Every day failed to load. First error: {skipped[0][1]}')

## 7. Keep history tidy

With `RETAIN_DAYS = 0` every day is kept. Otherwise days older than `RETAIN_DAYS` are deleted. On a tenant with no activity yet, an empty table is created so the semantic model has something to read.

In [ ]:
if not spark.catalog.tableExists(OUTPUT_TABLE):
    spark.createDataFrame([], schema=SCHEMA).write.format('delta').mode('overwrite').saveAsTable(OUTPUT_TABLE)
    print(f'Created an empty {OUTPUT_TABLE}; it fills in as Microsoft publishes reports.')

if RETAIN_DAYS and RETAIN_DAYS > 0:
    cutoff = today - timedelta(days=int(RETAIN_DAYS))
    spark.sql(f"DELETE FROM {OUTPUT_TABLE} WHERE ActivityDate < DATE '{cutoff.isoformat()}'")
    print(f'Deleted days before {cutoff}.')

## 8. Summary

In [ ]:
tbl = spark.table(OUTPUT_TABLE)
stats = tbl.agg(
    F.count(F.lit(1)).alias('rows'),
    F.countDistinct('ActivityDate').alias('days'),
    F.countDistinct('UPN_Normalized').alias('people'),
    F.min('ActivityDate').alias('first'),
    F.max('ActivityDate').alias('last'),
).collect()[0]
print(f"✓ {OUTPUT_TABLE}: {stats['rows']:,} rows, {stats['people']:,} people, "
      f"{stats['days']} days ({stats['first']} to {stats['last']}).")
print(f'  This run wrote {len(written)} day(s); {len(skipped)} skipped and will be retried next run.')

if people_seen and concealed_seen / people_seen > 0.5:
    print('\n⚠ Most user names in the reports are concealed, so this activity cannot be matched to Copilot use.\n'
          '  In the Microsoft 365 admin center, go to Settings → Org settings → Reports and turn off\n'
          '  "Display concealed user, group, and site names in all reports". The next run reloads every\n'
          '  concealed day still in Microsoft\'s 28-day window with real names.')

(tbl.groupBy('ActivityDate')
    .agg(F.count(F.lit(1)).alias('people'), *[F.sum(c).alias(c) for c in ACTIVE])
    .orderBy(F.desc('ActivityDate'))
    .show(10, truncate=False))